# Using Tools with Claude

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv(override=True)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

### Define the tool functions

Let's define a tool function to get current date & time in a given format.

In [2]:
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be null")
    return datetime.now().strftime(date_format)

In [3]:
# some test calls
print(f"Default: {get_current_datetime()}")
print(f"Custom: {get_current_datetime('%d/%m/%Y %I:%M %p')}")

Default: 2026-09-07 13:18:14
Custom: 07/09/2026 01:18 PM


We will also need to create a JSON schema describing the tool call function & params. This can be generated using Claude AI. Following schema was generated by Claude AI, which we assign to a varible, named with the same name as the tool function and ending with `_schema`.

In [14]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

Now let's call Claude with this tool schema (JSON) and a user query. 

In [20]:
messages = []

messages.append(
    {"role": "user", "content": "What is the exact time formatted as HH:MM:SS?"}
)

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(response)

Message(id='msg_011CepAaV6jt19gFpmcCT3js', container=None, content=[ToolUseBlock(id='toolu_01BFGPs4rWU2Phg7qZ6pFun6', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=618, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))
